<h1 align="center">Play Memgraph with Langchain</h1>

In [ ]:
pip install langchain langchain-openai neo4j langchain-community --user

In [ ]:
import os

from langchain_community.chains.graph_qa.memgraph import MemgraphQAChain
from langchain_community.graphs import MemgraphGraph
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

url = os.environ.get("MEMGRAPH_URI", "bolt://localhost:7687")
username = os.environ.get("MEMGRAPH_USERNAME", "admin")
password = os.environ.get("MEMGRAPH_PASSWORD", "kafka")

graph = MemgraphGraph(
    url=url, username=username, password=password, refresh_schema=False
)

In [ ]:
# Drop graph
graph.query("STORAGE MODE IN_MEMORY_ANALYTICAL")
graph.query("DROP GRAPH")
graph.query("STORAGE MODE IN_MEMORY_TRANSACTIONAL")

# Creating and executing the seeding query
query = """
    MERGE (g:Game {name: "Baldur's Gate 3"})
    WITH g, ["PlayStation 5", "Mac OS", "Windows", "Xbox Series X/S"] AS platforms,
            ["Adventure", "Role-Playing Game", "Strategy"] AS genres
    FOREACH (platform IN platforms |
        MERGE (p:Platform {name: platform})
        MERGE (g)-[:AVAILABLE_ON]->(p)
    )
    FOREACH (genre IN genres |
        MERGE (gn:Genre {name: genre})
        MERGE (g)-[:HAS_GENRE]->(gn)
    )
    MERGE (p:Publisher {name: "Larian Studios"})
    MERGE (g)-[:PUBLISHED_BY]->(p);
"""

graph.query(query)

In [ ]:
x="match (n) return n;"
graph.query(x)

In [ ]:
graph.refresh_schema()

In [ ]:
from neo4j import GraphDatabase

with GraphDatabase.driver(url, auth=(username, password)) as driver:
    driver.verify_connectivity()
    print("Connection established.")

    records, summary, keys = driver.execute_query( "MATCH (n) RETURN n")
    
    # Loop through results and do something with them
    for record in records:
        print(record.data())